# ChEMBL

ChEMBL is a massive dataset recoridng how strongly molecule X bings against protein Y.

It has been built up over 40+ years by curating data from published medicinal chemistry papers.

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd
from tqdm import tqdm
import time

# 1. Build the query (lazy, no API call yet)
activity = new_client.activity
results = activity.filter(
    target_chembl_id="CHEMBL279",
    standard_type__in=["IC50", "Ki", "Kd"],
    assay_type="B",
    pchembl_value__isnull=False,
    data_validity_comment__isnull=True,
).only([
    "molecule_chembl_id",
    "canonical_smiles",
    "pchembl_value",
    "standard_type",
    "standard_value",
    "standard_units",
    "assay_chembl_id",
    "target_chembl_id",
    "document_chembl_id",
])

# 2. Get the total count first (one fast API call)
print("Querying total count...")
t0 = time.time()
total = len(results)
print(f"Found {total:,} activity rows ({time.time()-t0:.1f}s)")

# 3. Iterate with a progress bar
rows = []
for record in tqdm(results, total=total, desc="Pulling activities"):
    rows.append(record)

df = pd.DataFrame(rows)
print(f"\nPulled {len(df):,} activity rows into DataFrame")

Querying total count...
Found 13,279 activity rows (0.0s)


Pulling activities: 100%|██████████| 13279/13279 [06:28<00:00, 34.20it/s]  


Pulled 13,279 activity rows into DataFrame


In [12]:
import pandas as pd
import numpy as np

# 1. Cast pchembl to float so we get real numeric stats
df["pchembl_value"] = pd.to_numeric(df["pchembl_value"], errors="coerce")
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

# 2. Drop rows missing SMILES or pchembl
before = len(df)
df = df.dropna(subset=["canonical_smiles", "pchembl_value"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with missing SMILES/pchembl")
print(f"Now: {len(df):,} rows")

# 3. Now look at real numeric stats
print("\npchembl_value distribution:")
print(df["pchembl_value"].describe())

print("\nstandard_type counts:")
print(df["standard_type"].value_counts())

Dropped 4 rows with missing SMILES/pchembl
Now: 13,275 rows

pchembl_value distribution:
count    13275.000000
mean         6.902634
std          1.169990
min          4.000000
25%          6.030000
50%          6.980000
75%          7.750000
max         10.700000
Name: pchembl_value, dtype: float64

standard_type counts:
standard_type
IC50    12530
Ki        561
Kd        184
Name: count, dtype: int64


In [13]:
from chembl_webresource_client.new_client import new_client
from tqdm import tqdm

assay = new_client.assay
assay_ids = df["assay_chembl_id"].unique().tolist()
print(f"Pulling metadata for {len(assay_ids):,} unique assays...")

assay_results = assay.filter(assay_chembl_id__in=assay_ids).only(
    ["assay_chembl_id", "confidence_score"]
)

assay_rows = []
for record in tqdm(assay_results, total=len(assay_ids), desc="Pulling assays"):
    assay_rows.append(record)

assay_df = pd.DataFrame(assay_rows)
print(f"Pulled {len(assay_df):,} assay records")

# Join onto df
df = df.merge(
    assay_df[["assay_chembl_id", "confidence_score"]],
    on="assay_chembl_id",
    how="left",
)
df["confidence_score"] = pd.to_numeric(df["confidence_score"], errors="coerce")

print(f"\nAfter join: {len(df):,} rows")
print("\nConfidence score distribution:")
print(df["confidence_score"].value_counts().sort_index())
print(f"\nRows with confidence >= 8: {(df['confidence_score'] >= 8).sum():,}")

Pulling metadata for 1,186 unique assays...


Pulling assays: 100%|██████████| 1186/1186 [00:41<00:00, 28.50it/s]

Pulled 1,186 assay records

After join: 13,275 rows

Confidence score distribution:
confidence_score
8    3327
9    9948
Name: count, dtype: int64

Rows with confidence >= 8: 13,275


In [15]:
from rdkit import Chem
from rdkit import RDLogger
from tqdm import tqdm

# Silence RDKit's verbose warnings during parsing
RDLogger.DisableLog("rdApp.*")

# 1. Canonicalize SMILES via RDKit
def canonicalize(smi):
    if not isinstance(smi, str):
        return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)

tqdm.pandas(desc="Canonicalizing SMILES")
df["rdkit_smiles"] = df["canonical_smiles"].progress_apply(canonicalize)

# 2. Drop rows where RDKit couldn't parse
before = len(df)
df = df.dropna(subset=["rdkit_smiles"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows RDKit couldn't parse")
print(f"Now: {len(df):,} rows")

# 3. Deduplicate to one row per molecule
mol_df = (
    df.groupby(["molecule_chembl_id", "rdkit_smiles"])
      .agg(
          pchembl_median=("pchembl_value", "median"),
          pchembl_std=("pchembl_value", "std"),
          pchembl_min=("pchembl_value", "min"),
          pchembl_max=("pchembl_value", "max"),
          n_measurements=("pchembl_value", "size"),
      )
      .reset_index()
)
print(f"\nUnique molecules: {len(mol_df):,}")

# 4. Label actives
mol_df["active"] = (mol_df["pchembl_median"] >= 6).astype(int)
print(f"Actives (pchembl >= 6):   {mol_df['active'].sum():,} ({100 * mol_df['active'].mean():.1f}%)")
print(f"Inactives (pchembl < 6):  {(1 - mol_df['active']).sum():,}")

# 5. Look at the multi-measurement molecules
print(f"\nMolecules with multiple measurements: {(mol_df['n_measurements'] > 1).sum():,}")
print("Median std across multi-measurement molecules:",
      mol_df.loc[mol_df['n_measurements'] > 1, 'pchembl_std'].median())

Canonicalizing SMILES: 100%|██████████| 13275/13275 [00:02<00:00, 4776.44it/s]

Dropped 0 rows RDKit couldn't parse
Now: 13,275 rows

Unique molecules: 9,912
Actives (pchembl >= 6):   7,678 (77.5%)
Inactives (pchembl < 6):  2,234

Molecules with multiple measurements: 2,232
Median std across multi-measurement molecules: 0.19091883092036785


# Split by Scaffold

In [16]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from tqdm import tqdm

def get_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except Exception:
        return None

tqdm.pandas(desc="Computing scaffolds")
mol_df["scaffold"] = mol_df["rdkit_smiles"].progress_apply(get_scaffold)

# Some scaffolds come back as empty strings (acyclic molecules — no rings)
mol_df["scaffold"] = mol_df["scaffold"].replace("", "ACYCLIC")

print(f"\nUnique scaffolds: {mol_df['scaffold'].nunique():,}")
print(f"\nTop 10 most common scaffolds:")
print(mol_df["scaffold"].value_counts().head(10))

# Scaffold size distribution
scaffold_sizes = mol_df["scaffold"].value_counts()
print(f"\nSingleton scaffolds (only 1 molecule): {(scaffold_sizes == 1).sum():,}")
print(f"Scaffolds with 2-5 molecules:           {((scaffold_sizes >= 2) & (scaffold_sizes <= 5)).sum():,}")
print(f"Scaffolds with 6-20 molecules:          {((scaffold_sizes >= 6) & (scaffold_sizes <= 20)).sum():,}")
print(f"Scaffolds with >20 molecules:           {(scaffold_sizes > 20).sum():,}")
print(f"Largest scaffold size:                  {scaffold_sizes.max():,}")

Computing scaffolds: 100%|██████████| 9912/9912 [00:02<00:00, 3523.69it/s]


Unique scaffolds: 3,770

Top 10 most common scaffolds:
scaffold
c1ccc(Nc2ncnc3ccccc23)cc1                          300
c1ccc(Nc2ccc3ncc(-c4cn[nH]c4)nc3c2)cc1             253
O=C1Nc2ccccc2Nc2nccc(Nc3ccccc3)c21                  84
O=C(Nc1ccccc1)Nc1ccc(Oc2ccnc(NC(=O)C3CC3)c2)cc1     81
O=C(Nc1ccccc1)Nc1ccc(Oc2ccnc(-c3ccc[nH]3)c2)cc1     76
c1ccc(Nc2ncnc3cc(OCC4CNCCO4)ccc23)cc1               70
c1ccc(Nc2ncccn2)cc1                                 65
c1ccc(-c2n[nH]c3ncncc23)cc1                         64
c1ccc(-c2cnc3cc(-c4ncccn4)ccn23)cc1                 59
c1ccc(N(CC2CC2)c2ccc3ncc(-c4cn[nH]c4)nc3c2)cc1      57
Name: count, dtype: int64

Singleton scaffolds (only 1 molecule): 2,608
Scaffolds with 2-5 molecules:           853
Scaffolds with 6-20 molecules:          255
Scaffolds with >20 molecules:           54
Largest scaffold size:                  300


In [17]:
import numpy as np

rng = np.random.default_rng(seed=42)

# Restrict to actives for the retrieval task
actives_df = mol_df[mol_df["active"] == 1].copy().reset_index(drop=True)
print(f"Actives: {len(actives_df):,}")
print(f"Unique scaffolds among actives: {actives_df['scaffold'].nunique():,}")

# For each non-singleton scaffold, hold out ~20% as queries
QUERY_FRAC = 0.20
query_idx = []

for scaffold, group in actives_df.groupby("scaffold"):
    n = len(group)
    if n < 2:
        # Singleton — leave entirely in corpus
        continue
    # Hold out at least 1, at most n-1 (keep at least 1 in corpus per scaffold)
    n_query = max(1, int(round(n * QUERY_FRAC)))
    n_query = min(n_query, n - 1)
    sampled = rng.choice(group.index.values, size=n_query, replace=False)
    query_idx.extend(sampled.tolist())

actives_df["split"] = "corpus"
actives_df.loc[query_idx, "split"] = "query"

print(f"\nQuery actives:  {(actives_df['split'] == 'query').sum():,}")
print(f"Corpus actives: {(actives_df['split'] == 'corpus').sum():,}")

# Build the full corpus: corpus actives + ALL inactives
corpus_df = pd.concat([
    actives_df[actives_df["split"] == "corpus"],
    mol_df[mol_df["active"] == 0].assign(split="corpus"),
], ignore_index=True)

query_df = actives_df[actives_df["split"] == "query"].reset_index(drop=True)

print(f"\nFinal corpus:  {len(corpus_df):,} molecules ({corpus_df['active'].sum():,} active, {(1 - corpus_df['active']).sum():,} inactive)")
print(f"Final queries: {len(query_df):,} (all active)")
print(f"\nFor each query, the retriever ranks the corpus.")
print(f"Hits = corpus actives that share scaffold with the query (and ideally beyond).")

Actives: 7,678
Unique scaffolds among actives: 2,861

Query actives:  1,433
Corpus actives: 6,245

Final corpus:  8,479 molecules (6,245 active, 2,234 inactive)
Final queries: 1,433 (all active)

For each query, the retriever ranks the corpus.
Hits = corpus actives that share scaffold with the query (and ideally beyond).
